<a href="https://colab.research.google.com/github/Osaretin226/fatigue-predictor/blob/main/Real_Fatigue_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Download ONLY lightweight text/TSV files from ds004902 (71 participants)
!pip install -q openneuro-py pandas numpy scikit-learn

import os, glob
import pandas as pd
import openneuro

os.makedirs("ds004902_beh", exist_ok=True)

# Exclude heavy raw EEG binary files; download all lightweight TSV/JSON files automatically
openneuro.download(
    dataset="ds004902",
    target_dir="ds004902_beh",
    exclude=["*.set", "*.fdt", "*.edf", "*.fif", "*.bdf", "*.eeg", "*.vmrk", "*.vhdr"]
)

# List all downloaded TSV files and inspect their columns
tsv_files = sorted(glob.glob("ds004902_beh/**/*.tsv", recursive=True))
print(f"Downloaded {len(tsv_files)} TSV files.")
for f in tsv_files[:10]:
    print(" -", f)

# Inspect participants.tsv and the first subject's TSV file
if os.path.exists("ds004902_beh/participants.tsv"):
    p_df = pd.read_csv("ds004902_beh/participants.tsv", sep="\t")
    print("\nparticipants.tsv columns:", list(p_df.columns))
    print(p_df.head(3))

sub_tsvs = [f for f in tsv_files if "sub-" in os.path.basename(f)]
if sub_tsvs:
    sample = pd.read_csv(sub_tsvs[0], sep="\t")
    print(f"\nSample subject file ({os.path.basename(sub_tsvs[0])}) columns:", list(sample.columns))
    print(sample.head(5))


🌍 Preparing to download ds004902 …
📥 Checking 793 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds004902 (downloaded 793 files and 919.0 kB).
Downloaded 428 TSV files.
 - ds004902_beh/participants.tsv
 - ds004902_beh/sub-01/ses-1/eeg/sub-01_ses-1_electrodes.tsv
 - ds004902_beh/sub-01/ses-1/eeg/sub-01_ses-1_task-eyesclosed_channels.tsv
 - ds004902_beh/sub-01/ses-1/eeg/sub-01_ses-1_task-eyesopen_channels.tsv
 - ds004902_beh/sub-01/ses-2/beh/sub-01_ses-2_task-pvt_beh.tsv
 - ds004902_beh/sub-01/ses-2/eeg/sub-01_ses-2_electrodes.tsv
 - ds004902_beh/sub-01/ses-2/eeg/sub-01_ses-2_task-eyesclosed_channels.tsv
 - ds004902_beh/sub-01/ses-2/eeg/sub-01_ses-2_task-eyesopen_channels.tsv
 - ds004902_beh/sub-02/ses-1/beh/sub-02_ses-1_task-pvt_beh.tsv
 - ds004902_beh/sub-02/ses-1/eeg/sub-02_ses-1_electrodes.tsv

participants.tsv columns: ['participant_id', 'Gender', 'Age', 'SessionOrder', 'EEG_SamplingTime_Open_NS', 'EEG_SamplingTime_Closed_NS', 'EEG_SamplingTime_Open_SD', 'EEG_SamplingTime_Closed_SD', 'PVT_SamplingTime_NS', 'PVT_SamplingTime_SD', 'PVT_item1_NS', 'PVT_item2

In [ ]:
# CELL 2: Parse Trial-Level PVT Reaction Times & Run 5-Fold GroupKFold CV
import os, glob
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 1. Load participants.tsv to map SessionOrder accurately (NS->SD vs SD->NS)
p_df = pd.read_csv("ds004902_beh/participants.tsv", sep="\t").set_index("participant_id")

pvt_files = sorted(glob.glob("ds004902_beh/sub-*/ses-*/beh/*_task-pvt_beh.tsv"))
print(f"Found {len(pvt_files)} PVT behavioral files.")

# Inspect first PVT file structure
sample_pvt = pd.read_csv(pvt_files[0], sep="\t")
print("PVT file columns:", list(sample_pvt.columns))
print(sample_pvt.head(3))

# 2. Extract trial-level reaction times per participant & session
session_rows = []
for filepath in pvt_files:
    parts = filepath.split(os.sep)
    sub_id = [p for p in parts if p.startswith("sub-")][0]
    ses_str = [p for p in parts if p.startswith("ses-")][0]  # 'ses-1' or 'ses-2'

    # Determine true condition from SessionOrder ('NS->SD' or 'SD->NS')
    order = str(p_df.loc[sub_id, "SessionOrder"]) if sub_id in p_df.index else "NS->SD"
    if order == "SD->NS":
        session_label = 1 if ses_str == "ses-1" else 0
    else:
        session_label = 0 if ses_str == "ses-1" else 1

    tsv = pd.read_csv(filepath, sep="\t")
    # Locate reaction time column dynamically
    rt_cols = [c for c in tsv.columns if 'rt' in c.lower() or 'response' in c.lower()]
    rt_col = rt_cols[0] if rt_cols else tsv.columns[-1]

    rts = pd.to_numeric(tsv[rt_col], errors='coerce').dropna()
    rts = rts[rts > 0]  # Remove missed/invalid zero entries
    rts_ms = rts * 1000 if rts.median() < 10 else rts  # Convert seconds to ms if needed

    if len(rts_ms) >= 5:
        session_rows.append({
            "participant_id": sub_id,
            "session_label": session_label,
            "mean_rt": rts_ms.mean(),
            "rt_variability": rts_ms.std(),
            "lapse_rate": (rts_ms > 500).mean(),
            "time_on_task_slope": np.polyfit(range(len(rts_ms)), rts_ms, 1)[0]
        })

features = pd.DataFrame(session_rows)
features.to_csv("pvt_session_features_71_subjects.csv", index=False)
print(f"\nCompiled {len(features)} valid sessions across {features['participant_id'].nunique()} participants.")
print("Condition balance (0=Normal Sleep, 1=Sleep Deprived):\n", features['session_label'].value_counts())

# 3. Participant-Grouped 5-Fold Cross-Validation
X = features[['mean_rt', 'rt_variability', 'lapse_rate', 'time_on_task_slope']]
y = features['session_label']
groups = features['participant_id']

cv = GroupKFold(n_splits=5)

dummy_pipe = DummyClassifier(strategy="most_frequent")
dummy_scores = cross_validate(dummy_pipe, X, y, groups=groups, cv=cv, scoring=['accuracy', 'roc_auc'])

rt_only_pipe = make_pipeline(StandardScaler(), LogisticRegression())
rt_scores = cross_validate(rt_only_pipe, X[['mean_rt']], y, groups=groups, cv=cv, scoring=['accuracy', 'roc_auc'])

full_pipe = make_pipeline(StandardScaler(), LogisticRegression())
full_scores = cross_validate(full_pipe, X, y, groups=groups, cv=cv, scoring=['accuracy', 'roc_auc'])

print("\n=== HONEST 5-FOLD PARTICIPANT-GROUPED RESULTS ===")
print(f"Majority Baseline   | Acc: {dummy_scores['test_accuracy'].mean():.2f} ± {dummy_scores['test_accuracy'].std():.2f} | AUC: {dummy_scores['test_roc_auc'].mean():.2f} ± {dummy_scores['test_roc_auc'].std():.2f}")
print(f"Mean RT Only        | Acc: {rt_scores['test_accuracy'].mean():.2f} ± {rt_scores['test_accuracy'].std():.2f} | AUC: {rt_scores['test_roc_auc'].mean():.2f} ± {rt_scores['test_roc_auc'].std():.2f}")
print(f"Full 4-Feature ML   | Acc: {full_scores['test_accuracy'].mean():.2f} ± {full_scores['test_accuracy'].std():.2f} | AUC: {full_scores['test_roc_auc'].mean():.2f} ± {full_scores['test_roc_auc'].std():.2f}")


Found 67 PVT behavioral files.
PVT file columns: ['trial', 'response']
   trial  response
0      1       306
1      2       305
2      3       260

Compiled 67 valid sessions across 38 participants.
Condition balance (0=Normal Sleep, 1=Sleep Deprived):
 session_label
1    38
0    29
Name: count, dtype: int64

=== HONEST 5-FOLD PARTICIPANT-GROUPED RESULTS ===
Majority Baseline   | Acc: 0.57 ± 0.03 | AUC: 0.50 ± 0.00
Mean RT Only        | Acc: 0.54 ± 0.06 | AUC: 0.44 ± 0.07
Full 4-Feature ML   | Acc: 0.51 ± 0.07 | AUC: 0.55 ± 0.11
